# MedGemma 4B ESI Fine-Tuning

QLoRA fine-tuning and held-out evaluation on authorized MIMIC-IV-ED triage
data. The model is trained to return one ESI label and evaluated by directly
scoring labels 1-5. This notebook is for research only.

## 1. Configuration

Select a profile and enable only the stage you want to run. Training requires
CUDA. Base and adapter evaluation use the validation split for temperature
calibration, then evaluate the untouched test split.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "prepare_training_data.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the oss_model_clinical_triage_demo repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

WindowsPath('C:/Users/money/OneDrive/Documents/GitHub/oss_model_clinical_triage_demo')

In [2]:
MODEL_ID = "google/medgemma-1.5-4b-it"
LOCAL_FILES_ONLY = True

PROFILE = "mimic_smoke"  # smoke, larger_smoke, mimic_smoke, balanced_large, or full
REVIEW_CONFIDENCE_THRESHOLD = 0.60
FORCE_GPU_ONLY_LOAD = True

RUN_BASE_EVALUATION = False
RUN_TRAINING = False
RUN_ADAPTER_EVALUATION = False

PROFILES = {
    "smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_smoke_label_scoring",
        "train_limit": 20,
        "validation_limit": 10,
        "test_limit": 10,
        "max_steps": 1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 1,
        "max_length": 256,
        "learning_rate": 2e-4,
        "lora_r": 4,
        "lora_alpha": 8,
        "eval_strategy": "no",
        "save_strategy": "no",
        "eval_steps": None,
        "save_steps": None,
    },
    "larger_smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_larger_smoke_label_scoring",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 20,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "eval_steps": 10,
        "save_steps": 10,
    },
    "mimic_smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_mimic_smoke_label_scoring",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 60,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "eval_steps": 20,
        "save_steps": 20,
    },
    "balanced_large": {
        "data_dir": "data/finetune_mimic_balanced_large",
        "output_dir": "outputs/notebook_mimic_balanced_large_label_scoring",
        "train_limit": None,
        "validation_limit": None,
        "test_limit": None,
        "max_steps": -1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "eval_steps": 100,
        "save_steps": 100,
    },
    "full": {
        "data_dir": "data/finetune_mimic",
        "output_dir": "outputs/notebook_mimic_full_label_scoring",
        "train_limit": None,
        "validation_limit": 500,
        "test_limit": 1000,
        "max_steps": -1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "eval_steps": None,
        "save_steps": None,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"Unknown profile: {PROFILE}")

CONFIG = dict(PROFILES[PROFILE])
CONFIG["review_confidence_threshold"] = REVIEW_CONFIDENCE_THRESHOLD
CONFIG["force_gpu_only_load"] = FORCE_GPU_ONLY_LOAD
DATA_DIR = PROJECT_ROOT / CONFIG["data_dir"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["output_dir"]
ADAPTER_DIR = OUTPUT_DIR / "adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.Series(CONFIG, name=PROFILE)

data_dir                                        data/finetune_mimic_smoke
output_dir                     outputs/notebook_mimic_smoke_label_scoring
train_limit                                                           500
validation_limit                                                      100
test_limit                                                            100
max_steps                                                              60
epochs                                                                  1
batch_size                                                              1
gradient_accumulation                                                   8
max_length                                                            512
learning_rate                                                      0.0002
lora_r                                                                  8
lora_alpha                                                             16
eval_strategy                         

## 2. Data

Existing prepared files used a JSON-output instruction. They are normalized in
memory to the single-label format used by both training and evaluation. Patient
separation is checked after loading.

In [3]:
OLD_INSTRUCTION = "Return only JSON with the key predicted_esi_level."
NEW_INSTRUCTION = "Return only one ESI level: 1, 2, 3, 4, or 5."


def normalize_record(record: dict) -> dict:
    user_message = dict(record["messages"][0])
    user_message["content"] = user_message["content"].replace(
        OLD_INSTRUCTION,
        NEW_INSTRUCTION,
    )
    return {
        **record,
        "messages": [
            user_message,
            {"role": "assistant", "content": str(record["label"])},
        ],
    }


def load_jsonl(path: Path) -> list[dict]:
    return [
        normalize_record(json.loads(line))
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def apply_limit(records: list[dict], limit: int | None) -> list[dict]:
    return records if limit is None else records[:limit]


train_records = apply_limit(load_jsonl(DATA_DIR / "train.jsonl"), CONFIG["train_limit"])
validation_records = apply_limit(
    load_jsonl(DATA_DIR / "validation.jsonl"),
    CONFIG["validation_limit"],
)
test_records = apply_limit(load_jsonl(DATA_DIR / "test.jsonl"), CONFIG["test_limit"])

subject_sets = {
    "train": {record["subject_id"] for record in train_records},
    "validation": {record["subject_id"] for record in validation_records},
    "test": {record["subject_id"] for record in test_records},
}
assert subject_sets["train"].isdisjoint(subject_sets["validation"])
assert subject_sets["train"].isdisjoint(subject_sets["test"])
assert subject_sets["validation"].isdisjoint(subject_sets["test"])
assert all(record["messages"][1]["content"] in {"1", "2", "3", "4", "5"} for record in train_records)

display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_records), len(validation_records), len(test_records)],
    "subjects": [len(subject_sets[name]) for name in ("train", "validation", "test")],
}))
print("No subject overlap. Training and evaluation use the same label format.")

,split,rows,subjects
0,train,500,496
1,validation,100,95
2,test,100,100


No subject overlap. Training and evaluation use the same label format.


## 3. Metrics

ESI 1 is most urgent. A prediction above the label is under-triage; a
prediction below it is over-triage. Automatic coverage excludes cases below
the configurable review threshold.

In [4]:
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from tqdm.auto import tqdm


def test_fingerprint(records: list[dict]) -> str:
    keys = [
        f"{record['subject_id']}:{record['stay_id']}:{record['label']}"
        for record in records
    ]
    return hashlib.sha256("|".join(keys).encode("utf-8")).hexdigest()


def compute_clinical_metrics(predictions: list[dict]) -> dict:
    actual = np.array([row["actual"] for row in predictions], dtype=int)
    predicted = np.array([row["prediction"] for row in predictions], dtype=int)
    review = np.array([row["needs_human_review"] for row in predictions], dtype=bool)
    automatic = ~review
    error = predicted - actual

    recalls = recall_score(
        actual,
        predicted,
        labels=[1, 2, 3, 4, 5],
        average=None,
        zero_division=0,
    )
    automatic_count = int(automatic.sum())
    return {
        "examples": int(len(actual)),
        "accuracy": float((predicted == actual).mean()),
        "macro_f1": float(f1_score(
            actual,
            predicted,
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0,
        )),
        "under_triage_rate": float((error > 0).mean()),
        "over_triage_rate": float((error < 0).mean()),
        "severe_under_triage_rate": float((
            np.isin(actual, [1, 2]) & np.isin(predicted, [4, 5])
        ).mean()),
        "automatic_coverage": float(automatic.mean()),
        "automatic_accuracy": (
            float((predicted[automatic] == actual[automatic]).mean())
            if automatic_count else None
        ),
        "human_review_rate": float(review.mean()),
        "average_prediction_confidence": float(np.mean([
            row["prediction_confidence"] for row in predictions
        ])),
        "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        "recall_by_esi": {
            str(label): float(value)
            for label, value in zip([1, 2, 3, 4, 5], recalls)
        },
        "confusion_matrix_labels_1_to_5": confusion_matrix(
            actual,
            predicted,
            labels=[1, 2, 3, 4, 5],
        ).tolist(),
    }


def save_evaluation(name: str, predictions: list[dict], temperature: float) -> dict:
    result = {
        "model_id": MODEL_ID,
        "profile": PROFILE,
        "evaluation_method": "five_label_next_token_scoring",
        "test_fingerprint": test_fingerprint(test_records),
        "calibration": {
            "method": "validation_temperature_scaling",
            "temperature": temperature,
            "review_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        },
        "metrics": compute_clinical_metrics(predictions),
        "predictions": predictions,
    }
    path = OUTPUT_DIR / f"{name}.json"
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"Saved {path}")
    return result


def display_metrics(result: dict):
    metrics = result["metrics"]
    display(pd.DataFrame([{
        key: value
        for key, value in metrics.items()
        if key not in {"recall_by_esi", "confusion_matrix_labels_1_to_5"}
    }]))
    display(pd.DataFrame(
        metrics["confusion_matrix_labels_1_to_5"],
        index=[f"actual {level}" for level in range(1, 6)],
        columns=[f"predicted {level}" for level in range(1, 6)],
    ))
    display(pd.DataFrame.from_dict(
        metrics["recall_by_esi"],
        orient="index",
        columns=["recall"],
    ))

## 4. Model loading and five-label scoring

Evaluation performs one forward pass per record and compares only the next
tokens for labels 1-5.

In [5]:
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def release_model_memory():
    for name in (
        "base_model",
        "base_processor",
        "adapter_model",
        "adapter_processor",
        "trainer",
        "model",
        "processor",
    ):
        globals().pop(name, None)
    cleanup_cuda()


def first_model_device(model) -> torch.device:
    for device in getattr(model, "hf_device_map", {}).values():
        if device not in {"cpu", "disk"}:
            return torch.device(device)
    return next(model.parameters()).device


def cuda_compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def quantization_config():
    compute_dtype = cuda_compute_dtype()
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_storage=compute_dtype,
    )


def model_load_kwargs(for_training: bool = False) -> dict:
    if torch.cuda.is_available():
        return {
            "quantization_config": quantization_config(),
            "dtype": cuda_compute_dtype(),
            "attn_implementation": "eager",
            "device_map": {"": 0} if FORCE_GPU_ONLY_LOAD else "auto",
        }
    if for_training:
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    return {
        "dtype": torch.float32,
        "low_cpu_mem_usage": True,
        "attn_implementation": "eager",
    }


def load_inference_model(adapter_dir: Path | None = None):
    release_model_memory()
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=False),
    )
    if adapter_dir is not None:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, str(adapter_dir))
    model.eval()
    return model, processor


def get_label_token_ids(tokenizer, user_message: dict) -> list[int]:
    prefix_text = tokenizer.apply_chat_template(
        [user_message],
        add_generation_prompt=True,
        tokenize=False,
    )
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False).input_ids
    label_ids = []
    for label in range(1, 6):
        full_text = tokenizer.apply_chat_template(
            [user_message, {"role": "assistant", "content": str(label)}],
            add_generation_prompt=False,
            tokenize=False,
        )
        full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
        if full_ids[:len(prefix_ids)] != prefix_ids:
            raise ValueError("Assistant label does not share the expected prompt prefix.")
        suffix = full_ids[len(prefix_ids):]
        if not suffix:
            raise ValueError(f"No token found for ESI label {label}.")
        label_ids.append(suffix[0])
    if len(set(label_ids)) != 5:
        raise ValueError("ESI labels do not map to five distinct tokens.")
    return label_ids


def collect_label_scores(model, tokenizer, records: list[dict], description: str):
    label_ids = get_label_token_ids(tokenizer, records[0]["messages"][0])
    device = first_model_device(model)
    rows = []
    for record in tqdm(records, desc=description, unit="case"):
        prompt_text = tokenizer.apply_chat_template(
            [record["messages"][0]],
            add_generation_prompt=True,
            tokenize=False,
        )
        inputs = tokenizer(
            prompt_text,
            add_special_tokens=False,
            return_tensors="pt",
            return_dict=True,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, label_ids].float().cpu()
        rows.append({
            "subject_id": record["subject_id"],
            "stay_id": record["stay_id"],
            "actual": record["label"],
            "label_logits": logits.tolist(),
        })
    return rows


def fit_temperature(validation_scores: list[dict]) -> float:
    logits = torch.tensor([row["label_logits"] for row in validation_scores])
    targets = torch.tensor([row["actual"] - 1 for row in validation_scores])
    temperatures = torch.logspace(-1, 1, steps=81)
    losses = torch.stack([
        torch.nn.functional.cross_entropy(logits / temperature, targets)
        for temperature in temperatures
    ])
    return float(temperatures[losses.argmin()].item())


def finalize_predictions(scores: list[dict], temperature: float) -> list[dict]:
    predictions = []
    for row in scores:
        probabilities = torch.softmax(
            torch.tensor(row["label_logits"]) / temperature,
            dim=0,
        )
        prediction = int(probabilities.argmax().item()) + 1
        confidence = float(probabilities.max().item())
        review = confidence < REVIEW_CONFIDENCE_THRESHOLD
        predictions.append({
            "subject_id": row["subject_id"],
            "stay_id": row["stay_id"],
            "actual": row["actual"],
            "prediction": prediction,
            "decision": "uncertain" if review else prediction,
            "prediction_confidence": confidence,
            "needs_human_review": review,
            "review_reason": "low_candidate_probability" if review else "",
            "class_probabilities": {
                str(label): float(probabilities[label - 1].item())
                for label in range(1, 6)
            },
        })
    return predictions


def evaluate_model(model, tokenizer, description: str):
    validation_scores = collect_label_scores(
        model,
        tokenizer,
        validation_records,
        f"{description} validation",
    )
    temperature = fit_temperature(validation_scores)
    print(f"Validation temperature: {temperature:.3f}")
    test_scores = collect_label_scores(
        model,
        tokenizer,
        test_records,
        f"{description} test",
    )
    return finalize_predictions(test_scores, temperature), temperature

## 5. Base-model evaluation

In [6]:
BASE_EVALUATION_PATH = OUTPUT_DIR / "base_evaluation.json"

if RUN_BASE_EVALUATION:
    base_model, base_processor = load_inference_model()
    base_predictions, base_temperature = evaluate_model(
        base_model,
        base_processor.tokenizer,
        "Base MedGemma",
    )
    base_result = save_evaluation(
        "base_evaluation",
        base_predictions,
        base_temperature,
    )
    display_metrics(base_result)
    release_model_memory()
elif BASE_EVALUATION_PATH.exists():
    base_result = json.loads(BASE_EVALUATION_PATH.read_text(encoding="utf-8"))
    display_metrics(base_result)
else:
    base_result = None
    print("Set RUN_BASE_EVALUATION = True and rerun this cell.")

Set RUN_BASE_EVALUATION = True and rerun this cell.


## 6. QLoRA training

Only language-model LoRA adapters are trained. The target completion is one
label (`1` through `5`), matching evaluation. Step-based profiles evaluate only
at configured intervals and reload the checkpoint with the best validation
loss before the final adapter is saved.

In [7]:
from datasets import Dataset


def to_sft_dataset(records: list[dict]) -> Dataset:
    return Dataset.from_list([
        {
            "prompt": [record["messages"][0]],
            "completion": [
                {"role": "assistant", "content": str(record["label"])}
            ],
        }
        for record in records
    ])


train_dataset = to_sft_dataset(train_records)
validation_dataset = to_sft_dataset(validation_records)
print(f"Training records: {len(train_dataset)}")
print(f"Validation records: {len(validation_dataset)}")

Training records: 500
Validation records: 100


In [8]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


LANGUAGE_LORA_TARGETS = (
    r".*language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|"
    r"mlp\.(?:gate_proj|up_proj|down_proj))$"
)


def hardware_snapshot() -> dict:
    snapshot = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(),
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        snapshot["gpu"] = {
            "name": properties.name,
            "total_vram_gib": properties.total_memory / 1024**3,
            "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
        }
    return snapshot


if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")

    release_model_memory()
    torch.cuda.reset_peak_memory_stats()
    training_started = time.perf_counter()

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=True),
    )
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0.05,
        bias="none",
        target_modules=LANGUAGE_LORA_TARGETS,
        task_type="CAUSAL_LM",
    )

    training_kwargs = {
        "output_dir": str(OUTPUT_DIR),
        "num_train_epochs": CONFIG["epochs"],
        "max_steps": CONFIG["max_steps"],
        "per_device_train_batch_size": CONFIG["batch_size"],
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": CONFIG["gradient_accumulation"],
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "learning_rate": CONFIG["learning_rate"],
        "warmup_ratio": 0.03,
        "max_grad_norm": 0.3,
        "optim": "paged_adamw_8bit",
        "bf16": torch.cuda.is_bf16_supported(),
        "fp16": not torch.cuda.is_bf16_supported(),
        "max_length": CONFIG["max_length"],
        "completion_only_loss": True,
        "eval_strategy": CONFIG["eval_strategy"],
        "save_strategy": CONFIG["save_strategy"],
        "logging_steps": 1,
        "logging_first_step": True,
        "report_to": "none",
        "seed": 42,
        "dataset_num_proc": 1,
        "save_total_limit": 2,
    }
    if CONFIG["eval_strategy"] == "steps":
        training_kwargs["eval_steps"] = CONFIG["eval_steps"]
        training_kwargs["save_steps"] = CONFIG["save_steps"]
    if CONFIG["eval_strategy"] != "no":
        training_kwargs.update({
            "load_best_model_at_end": True,
            "metric_for_best_model": "eval_loss",
            "greater_is_better": False,
        })

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(**training_kwargs),
        train_dataset=train_dataset,
        eval_dataset=(
            validation_dataset
            if CONFIG["eval_strategy"] != "no"
            else None
        ),
        peft_config=peft_config,
        processing_class=processor.tokenizer,
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in trainer.model.parameters()
        if parameter.requires_grad
    )
    assert not any(
        "vision_tower" in name and parameter.requires_grad
        for name, parameter in trainer.model.named_parameters()
    )
    print(f"Trainable parameters: {trainable_parameters:,}")

    train_output = trainer.train()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(ADAPTER_DIR))
    processor.tokenizer.save_pretrained(ADAPTER_DIR)

    training_report = {
        **train_output.metrics,
        "profile": PROFILE,
        "model_id": MODEL_ID,
        "train_records": len(train_records),
        "validation_records": len(validation_records),
        "trainable_parameters": trainable_parameters,
        "elapsed_seconds": time.perf_counter() - training_started,
        "config": CONFIG,
        "hardware": hardware_snapshot(),
    }
    (OUTPUT_DIR / "training_metrics.json").write_text(
        json.dumps(training_report, indent=2),
        encoding="utf-8",
    )
    print(f"Adapter saved to {ADAPTER_DIR}")
    release_model_memory()
else:
    print("Set RUN_TRAINING = True and rerun this cell.")

Set RUN_TRAINING = True and rerun this cell.


## 7. Adapter evaluation

In [9]:
ADAPTER_EVALUATION_PATH = OUTPUT_DIR / "adapter_evaluation.json"

if RUN_ADAPTER_EVALUATION:
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"No adapter found at {ADAPTER_DIR}")

    adapter_model, adapter_processor = load_inference_model(ADAPTER_DIR)
    adapter_predictions, adapter_temperature = evaluate_model(
        adapter_model,
        adapter_processor.tokenizer,
        "Fine-tuned MedGemma",
    )
    adapter_result = save_evaluation(
        "adapter_evaluation",
        adapter_predictions,
        adapter_temperature,
    )
    display_metrics(adapter_result)
    release_model_memory()
elif ADAPTER_EVALUATION_PATH.exists():
    adapter_result = json.loads(ADAPTER_EVALUATION_PATH.read_text(encoding="utf-8"))
    display_metrics(adapter_result)
else:
    adapter_result = None
    print("Train an adapter, then set RUN_ADAPTER_EVALUATION = True.")

Train an adapter, then set RUN_ADAPTER_EVALUATION = True.


## 8. Base and adapter comparison

In [10]:
if BASE_EVALUATION_PATH.exists() and ADAPTER_EVALUATION_PATH.exists():
    base_result = json.loads(BASE_EVALUATION_PATH.read_text(encoding="utf-8"))
    adapter_result = json.loads(ADAPTER_EVALUATION_PATH.read_text(encoding="utf-8"))
    if base_result["test_fingerprint"] != adapter_result["test_fingerprint"]:
        raise ValueError("Base and adapter evaluations used different test records.")

    metric_names = [
        "accuracy",
        "macro_f1",
        "under_triage_rate",
        "over_triage_rate",
        "severe_under_triage_rate",
        "automatic_coverage",
        "automatic_accuracy",
        "human_review_rate",
    ]
    comparison = pd.DataFrame({
        "metric": metric_names,
        "base": [base_result["metrics"][name] for name in metric_names],
        "fine_tuned": [adapter_result["metrics"][name] for name in metric_names],
    })
    comparison["change"] = comparison["fine_tuned"] - comparison["base"]
    display(comparison)

    display(pd.DataFrame({
        "base_recall": base_result["metrics"]["recall_by_esi"],
        "fine_tuned_recall": adapter_result["metrics"]["recall_by_esi"],
    }))
else:
    print("Run both evaluations to create a matched comparison.")

Run both evaluations to create a matched comparison.


## Interpretation limits

- The smoke profiles use balanced samples for debugging and model comparison;
  they do not estimate performance under natural MIMIC-IV-ED prevalence.
- The review probability is temperature-scaled on validation data, but the
  `0.60` policy threshold is not clinically validated.
- A larger untouched test cohort, repeated seeds, and external validation are
  required before making clinical claims.